# Scratch research
Imports only: anything worth keeping moves to `strategy.py`, then to `core_quant` once a second strategy needs it.

In [3]:
import pandas as pd
import akshare as ak
%load_ext autoreload
%autoreload 2
from strategies.dce_curve_structure.strategy import load, make_evaluator
from core_quant import validation, plots, feature_registry, utilities
from core_quant.walk_forward import run_validation

cfg = load()
cfg

Config(data_path='data/processed/<file>.parquet', split=SplitConfig(development_start=datetime.date(2020, 1, 1), holdout_start=datetime.date(2024, 1, 1), holdout_end=None), strategy=StrategyConfig(lookback=20, threshold=2.0, cost_bps=1.0), validation=ValidationConfig(n_folds=6, periods_per_year=252, grid={'lookback': [10, 20, 40], 'threshold': [1.5, 2.0, 2.5]}, heatmaps=[['lookback', 'threshold']]), gates=GateConfig(min_mean_sharpe=0.5, min_pct_folds_positive=0.6, min_trades=100, min_wf_oos_sharpe=0.3))

In [ ]:
# evaluate, dates = make_evaluator(cfg)
# out = run_validation(cfg, evaluate, dates, STRATEGY_DIR)
# out['summary'].head(20)

In [36]:
# Get Akshare data for DCE futures curves
import time
# column names
OHLCV_FIELDS = ["datetime", "open", "high", "low", "close", "volume", "hold"]
# build func
def fetch_sina_ohlcv(symbols: dict[str, str], period: int = 30):
    frames = []
    for sym, name in symbols.items():
        raw_df = ak.futures_zh_min_sina(symbol=sym, period=period)
        #raw_df["datetime"] = pd.to_datetime(raw_df["datetime"])
        #raw_df.set_index("datetime", inplace=True)
        raw_df = raw_df.assign(symbol=sym, name=name)
        frames.append(raw_df[["symbol", "name"] + OHLCV_FIELDS])
    return pd.concat(frames, ignore_index=True).sort_values(["datetime", "symbol"])

SYMBOLS = {
    "A0": "bean1",
    "B0": "bean2",
    "M0": "meal",
    "Y0": "oil",
}


def build_dce_curve(products: list[str], years: range, months: list[int]) -> pd.DataFrame:
    all_contracts = []
    for prod in products:
        for year in years:
            for month in months:
                symbol = f"{prod}{year:02d}{month:02d}"
                try:
                    df = ak.futures_zh_daily_sina(symbol=symbol)
                    if df is not None and not df.empty:
                        df["symbol"] = symbol
                        df["product"] = prod
                        df["contract_yymm"] = f"20{year:02d}-{month:02d}"
                        all_contracts.append(df)
                except Exception as e:
                    print(f"Skipping {symbol}: {e}")

                time.sleep(0.3)

    if not all_contracts:
        return pd.DataFrame()

    panel_df = pd.concat(all_contracts, ignore_index=True)
    panel_df["date"] = pd.to_datetime(panel_df["date"])
    return panel_df


In [53]:
products = ["M", "Y", "A", "B", "C"]
years = range(20, 28)
months = [1, 5, 9]
panel_df = build_dce_curve(products=products, years=years, months=months)
panel_df.head()

,date,open,high,low,close,volume,hold,settle,symbol,product,contract_yymm
0,2019-01-16,2639.0,2666.0,2622.0,2631.0,40642,34906,2647.0,M2001,M,2020-01
1,2019-01-17,2635.0,2657.0,2631.0,2647.0,10574,41624,2643.0,M2001,M,2020-01
2,2019-01-18,2646.0,2687.0,2642.0,2682.0,11216,41758,2674.0,M2001,M,2020-01
3,2019-01-21,2693.0,2703.0,2678.0,2696.0,9858,44968,2689.0,M2001,M,2020-01
4,2019-01-22,2689.0,2700.0,2673.0,2687.0,3932,44402,2688.0,M2001,M,2020-01


In [60]:
pivot = panel_df.pivot_table(index="date", columns="product", values="close")
pivot

product,A,B,C,M,Y
date,,,,,
2019-01-02,3500.000000,NaN,NaN,NaN,NaN
2019-01-03,3508.000000,NaN,NaN,NaN,NaN
2019-01-04,3509.000000,NaN,NaN,NaN,NaN
2019-01-07,3510.000000,NaN,NaN,NaN,NaN
2019-01-08,3504.000000,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2026-11-12,5247.666667,3726.000000,2242.666667,2968.000000,8531.000000
2026-11-13,5240.000000,3734.000000,2243.333333,2973.666667,8546.333333
2026-11-16,5026.000000,3699.000000,2228.666667,2948.666667,8521.333333


In [16]:
df = fetch_sina_ohlcv(symbols=SYMBOLS, period=30)
df["datetime"] = pd.to_datetime(df["datetime"])
df.head()

,symbol,name,datetime,open,high,low,close,volume,hold
3069,Y0,oil,2026-07-13 15:00:00,8557.0,8569.0,8557.0,8568.0,68,583284
3070,Y0,oil,2026-07-13 21:30:00,8592.0,8594.0,8577.0,8582.0,31517,583442
0,A0,bean1,2026-07-13 22:00:00,4713.0,4722.0,4711.0,4717.0,10518,264458
1023,B0,bean2,2026-07-13 22:00:00,3790.0,3792.0,3783.0,3788.0,2,152905
2046,M0,meal,2026-07-13 22:00:00,3056.0,3058.0,3048.0,3056.0,71288,1939068


In [26]:
wide = df.pivot(index="datetime", columns="symbol", values=OHLCV_FIELDS)
wide.columns = wide.columns.swaplevel(0, 1)
wide.columns.names = ["name", "field"]

wide

name,A0,B0,M0,Y0,A0,B0,M0,Y0,A0,B0,...,M0,Y0,A0,B0,M0,Y0,A0,B0,M0,Y0
field,datetime,datetime,datetime,datetime,open,open,open,open,high,high,...,close,close,volume,volume,volume,volume,hold,hold,hold,hold
datetime,,,,,,,,,,,,,,,,,,,,,
2026-07-13 15:00:00,NaT,NaT,NaT,2026-07-13 15:00:00,NaN,NaN,NaN,8557.0,NaN,NaN,...,NaN,8568.0,NaN,NaN,NaN,68,NaN,NaN,NaN,583284
2026-07-13 21:30:00,NaT,NaT,NaT,2026-07-13 21:30:00,NaN,NaN,NaN,8592.0,NaN,NaN,...,NaN,8582.0,NaN,NaN,NaN,31517,NaN,NaN,NaN,583442
2026-07-13 22:00:00,2026-07-13 22:00:00,2026-07-13 22:00:00,2026-07-13 22:00:00,2026-07-13 22:00:00,4713.0,3790.0,3056.0,8582.0,4722.0,3792.0,...,3056.0,8566.0,10518,2,71288,17338,264458,152905,1939068,583451
2026-07-13 22:30:00,2026-07-13 22:30:00,2026-07-13 22:30:00,2026-07-13 22:30:00,2026-07-13 22:30:00,4717.0,3788.0,3055.0,8567.0,4726.0,3800.0,...,3064.0,8590.0,26,8,13813,46,262515,152673,1938285,582989
2026-07-13 23:00:00,2026-07-13 23:00:00,2026-07-13 23:00:00,2026-07-13 23:00:00,2026-07-13 23:00:00,4725.0,3800.0,3064.0,8591.0,4729.0,3800.0,...,3064.0,8587.0,691,304,37424,471,258970,152792,1936211,581595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-11-18 11:15:00,2026-11-18 11:15:00,2026-11-18 11:15:00,2026-11-18 11:15:00,2026-11-18 11:15:00,5010.0,3813.0,3097.0,8557.0,5030.0,3813.0,...,3095.0,8539.0,26020,4043,55637,32483,346819,166033,1884403,431471
2026-11-18 13:45:00,2026-11-18 13:45:00,2026-11-18 13:45:00,2026-11-18 13:45:00,2026-11-18 13:45:00,5017.0,3806.0,3095.0,8539.0,5025.0,3807.0,...,3091.0,8535.0,11273,5294,279,19219,346070,166993,1882799,432118
